# KAT-VAD v2.2 — **round 7, batch 1: R0-probe + K1** (no trunk training)

Design: `core/docs/v2.2/KAT_VAD_PROPOSAL_v2.2.md` §R7 (D1–D14). Plan: `.project/plans/katvad-v2.2-two-tracks.md` (G1–G10).
R0-saved already ran locally (saved scores): **ΔT(`p_T2`) = 0.970** → ΔT is retired as a decision estimand (D14);
everything here is **reported**, except K1's soft stop.

| step | what | trains? | decides |
|---|---|---|---|
| 0 | setup; refuses an upload without `v22_gates` | — | — |
| 1 | preflight + stage (CLIP, V2-S, T2 v2 data dir, flow/v1 stats, CAP alignment) | no | — |
| 2 | **R0-probe**: ΔT / detrended ΔT / span of `m5`, `s`, probes `F0`, `F0+κ`, `p`-only (R0-e headroom), R0-d precursor share | probes only | nothing (reported) |
| 3 | **K1**: probe `F0` vs `F0 + stats23` (flow/v1, shifted causal), read on **T2-val only** | probes only | soft stop (D10) |
| 4 | record → paste `r0_probe_readout.md` + `k1_readout.md` to Claude | — | — |


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), data/ (v2 dataset dir), outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V22_BAR_MIN'), 'this upload predates the v2.2 order diagnostics -- re-upload branch v2'
assert (REPO / 'core/tools/v22_gates.py').is_file(), 'upload has no core/tools/v22_gates.py -- re-upload branch v2.2'
assert hasattr(constants, 'V22_K1_SOFT_STOP'), 'this upload predates round 7 -- re-upload branch v2.2'
PROPOSAL = REPO / 'core/docs/v2.2/KAT_VAD_PROPOSAL_v2.2.md'
assert '## R7. Round 7' in PROPOSAL.read_text(encoding='utf-8'), 'upload has no round-7 v2.2 proposal'

DATASET = constants.DADA_ORIGIN_DATASET                    # 'DADA2000_orig'
ENC     = constants.VIDEOMAE_ENCODER_S                     # A3's video stream
T2      = constants.DATA_ROOT / DATASET
CENSUS  = T2 / 'counts' / 'resolved_census.json'
ORIG    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
ANNOT   = ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME
T2_CLIP = constants.CLIP_CACHE_DIR / DATASET               # s8, one file per source
VIDEO   = constants.VIDEO_CACHE_DIR
DOTA    = constants.DATA_ROOT / 'DoTA'
DOTA_S1 = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'        # T2-train - T2-val windows
OUT     = constants.OUTPUT_ROOT / 'REPORTS' / 'v22_gates'
ALIGN   = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap' / 'dota_cap_alignment.json'
FLOW    = constants.FLOW_CACHE_DIR / DATASET                   # flow/v1 raw RAFT stats, read only
LOCAL   = Path('/content/v22_gates')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2_CLIP', 'VIDEO', 'DOTA_S1', 'V2_DATA', 'ANNOT', 'CENSUS', 'ALIGN', 'FLOW', 'OUT'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU -- fine, S2/S3 just run slower'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import torch

ENV = dict(os.environ)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def run(cmd, log=None):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    # Copy a Drive dir (only ``ids`` + its manifests when given) to VM disk once, bit-identical (C2).
    t0 = time.time()
    if ids is None:
        if not dst.exists():
            shutil.copytree(src, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst

## 1. Preflight + stage

In [ ]:
from core.data.v2_splits import load_split
from core.tools import dota_cap, e2d_probe, extract_video_features, v22_gates, v2_dataset

cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)          # sha1-checked; eval never loaded
assert len(cap_dev) == 569, len(cap_dev)
val_src = load_split(constants.V2_SPLIT_T2_VAL)
assert (V2_DATA / v2_dataset.MANIFEST_FILENAME).is_file(), f'no v2 dataset dir at {V2_DATA}'
meta = json.loads((V2_DATA / constants.META_FILENAME).read_text())
train_windows = json.loads((V2_DATA / constants.LABELS_TRAIN_FILENAME).read_text())
train_src = sorted({meta[w]['source'] for w in train_windows})
assert not set(train_src) & set(val_src), 'T2-val sources among the training windows'
census = json.loads(CENSUS.read_text())
assert all(v in census for v in train_src + val_src), 'a T2 source has no census entry'
assert ANNOT.is_file(), f'missing {ANNOT}'

S_T2, S_DOTA = e2d_probe.t2_video_dir(VIDEO, ENC), e2d_probe.dota_video_dir(VIDEO, ENC)
alignment = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
e2d_probe.check_manifest(S_DOTA, dota_cap.video_manifest(ENC, alignment))
e2d_probe.check_manifest(S_T2, extract_video_features.build_manifest(
    ENC, constants.FRAME_STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC]))
t2_ids = train_src + list(val_src)
for d, ids, what in ((T2_CLIP, t2_ids, 'CLIP T2'), (S_T2, t2_ids, 'V2-S T2'),
                     (DOTA_S1, cap_dev, 'CLIP DoTA_s1_ncc'), (S_DOTA, cap_dev, 'V2-S DoTA-CAP')):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{what}: {len(miss)} missing, e.g. {miss[:3]}'
assert ALIGN.is_file(), f'missing {ALIGN}'
miss = [v for v in t2_ids if not (FLOW / f'{v}{constants.FLOW_STATS_SUFFIX}').exists()]
assert not miss, f'flow/v1 stats: {len(miss)} T2 sources missing, e.g. {miss[:3]}'
print(f'preflight OK | train sources {len(train_src)} ({len(train_windows)} windows) | '
      f'T2-val sources {len(val_src)} | DoTA-CAP-dev {len(cap_dev)}')

st = LOCAL / 'stage'
L_T2_CLIP = stage(T2_CLIP, st / 'clip_t2', t2_ids)
L_VIDEO = st / 'video'
stage(S_T2, L_VIDEO / ENC / S_T2.name, t2_ids)
stage(S_DOTA, L_VIDEO / ENC / S_DOTA.name, cap_dev)
L_DOTA_S1 = stage(DOTA_S1, st / 'DoTA_s1_ncc', cap_dev)
L_V2 = stage(V2_DATA, st / 'corpus_v2')
L_FLOW = st / 'flow_v1'
L_FLOW.mkdir(parents=True, exist_ok=True)
for v in t2_ids:
    f = FLOW / f'{v}{constants.FLOW_STATS_SUFFIX}'
    if not (L_FLOW / f.name).exists():
        shutil.copy2(f, L_FLOW / f.name)

COMMON = ['--data-dir', L_V2, '--annotation', ANNOT, '--census', CENSUS,
          '--t2-clip-dir', L_T2_CLIP, '--video-root', L_VIDEO, '--dota-s1-dir', L_DOTA_S1,
          '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
          '--device', DEVICE, '--out-dir', OUT, '--alignment', ALIGN]
SAVED = REPO / 'outputs/v2/REPORTS/v22_gates' / v22_gates.readout_json('r0-saved')
if SAVED.is_file() and not (OUT / SAVED.name).exists():
    shutil.copy2(SAVED, OUT / SAVED.name)        # completes the (moot) futility read; optional
print('device', DEVICE)

## 2. R0-probe — reported only (≈ S3's cost: 15 probe fits)

ΔT is retired as a decision estimand (D14), so nothing here decides. The reads that matter:
**R0-e** (`p`-only conv probe = the headroom inside the probe, C-b) and **R0-d** (are `m5`'s off-span peaks precursors?).


In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.v22_gates', 'r0-probe', *COMMON], log=OUT / 'r0_probe_run.log')
print(f'R0-probe in {(time.time() - t0) / 60:.1f} min\n')
print((OUT / v22_gates.readout_md('r0-probe')).read_text())


## 3. K1 — global flow kill-switch (T2-val only; DoTA has no flow)

`F0 + stats23` vs `F0`, 10 probe fits. **Soft stop** if both Δ points (span, ΔT) < +0.005: full EMS extraction then
needs a written advisor go-ahead. A K1 null is weak evidence (D10): T2-val Δ intervals are ≈ ±0.01 wide.


In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.v22_gates', 'k1', *COMMON, '--flow-dir', L_FLOW], log=OUT / 'k1_run.log')
print(f'K1 in {(time.time() - t0) / 60:.1f} min\n')
print((OUT / v22_gates.readout_md('k1')).read_text())


## 4. Record — then stop

Paste `Thesis-V2/outputs/REPORTS/v22_gates/{r0_probe,k1}_readout.md` to Claude. Batch 2 (SC-1 DF8) does not depend on them.
